# Astrolabio: modelo conjunto de la cartera en T4

Entrena en una GPU T4 el modelo que pronostica el rendimiento del día siguiente de los activos de la cartera. Sigue el mismo protocolo que el entrenamiento de desarrollo en CPU: mismas variables, mismas redes, mismos hiperparámetros, regla elegida con VA simulando la cartera completa y los mismos cortes de fecha que `dev-cartera-2025-09-17-40`. **TE no se abre.**

**Cómo ejecutarlo.** En Colab elige *Entorno de ejecución > Cambiar tipo > GPU T4* y después *Ejecutar todo*. Si Colab no asigna una T4, el notebook se detiene antes de entrenar. Al terminar se descarga `resultados_cartera_t4.zip`; su carpeta `t4-cartera-…` va a `models/` del proyecto.

**Datos.** Las velas diarias de `data/cartera/` y el universo de `config/universo.json` se bajan del repositorio público, fijados a un commit: son exactamente los del proyecto.

## Entorno y evidencia de T4

In [ ]:
import json, os, platform, shutil, subprocess, sys, time, zipfile
from pathlib import Path

PRUEBA = os.environ.get('NB_PRUEBA') == '1'  # solo para probar el notebook sin GPU, con cuatro activos y dos épocas; no es evidencia válida
SEMILLA_GLOBAL = 11
os.makedirs('bot', exist_ok=True); os.makedirs('research', exist_ok=True); os.makedirs('resultados', exist_ok=True)
for ruta in ('bot/__init__.py', 'research/__init__.py'):
    Path(ruta).touch()
SALIDA = Path('resultados')
inicio_notebook = time.time()

In [ ]:
evidencia = dict(python=platform.python_version(), semilla=SEMILLA_GLOBAL, inicio=time.strftime('%Y-%m-%d %H:%M:%S'), prueba_sin_gpu=PRUEBA)
if PRUEBA:
    print('MODO PRUEBA: sin T4, con cuatro activos y dos épocas. Esta corrida NO sirve como evidencia de T4.')
    evidencia |= dict(gpu='ninguna', valido_para_entrega=False)
else:
    try:
        salida = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout
    except (FileNotFoundError, subprocess.CalledProcessError) as e:
        raise RuntimeError('No hay GPU en esta sesión. Elige Entorno de ejecución > Cambiar tipo > GPU T4 y vuelve a ejecutar.') from e
    print(salida)
    (SALIDA / 'nvidia_smi.txt').write_text(salida)
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    nombre = tf.config.experimental.get_device_details(gpus[0]).get('device_name', '') if gpus else ''
    print('GPU detectada:', nombre or 'ninguna')
    if 'T4' not in nombre:
        raise RuntimeError('No se asignó una T4: se detiene el entrenamiento. Reprograma la ejecución.')
    with tf.device('/GPU:0'):
        a = tf.random.normal((2000, 2000))
        operacion = tf.linalg.matmul(a, a)
    print('Operación ejecutada en:', operacion.device)
    tf.config.experimental.reset_memory_stats('GPU:0')
    evidencia |= dict(gpu=nombre, tensorflow=tf.__version__, dispositivo_operacion=operacion.device, valido_para_entrega=True)
import keras
evidencia |= dict(keras=keras.__version__, backend=keras.backend.backend())
print(evidencia)

### Código del proyecto

Estas celdas escriben los mismos módulos que usa el agente y el entrenamiento en CPU: las variables, la partición y la regla se calculan igual.

In [ ]:
%%writefile bot/data.py
"""Auditoría, limpieza y partición de velas diarias. Nada aquí mira el futuro ni inventa filas."""
import hashlib

import numpy as np
import pandas as pd

DIA_MS = 86_400_000
COLUMNAS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume']
PRECIOS = ['open', 'high', 'low', 'close']


class DataError(ValueError):
    """Datos que impiden continuar. El mensaje explica qué revisar."""


def fecha(ms):
    return pd.Timestamp(int(ms), unit='ms', tz='UTC').strftime('%Y-%m-%d')


def frame_from_rows(filas):
    """Convierte las filas de /api/v3/klines en un DataFrame, sin corregir nada todavía."""
    if not filas:
        raise DataError('Conjunto vacío: la fuente no devolvió velas. Revisa símbolo, intervalo y conexión.')
    if len(filas[0]) < len(COLUMNAS):
        raise DataError(f'Fila con {len(filas[0])} columnas; se esperaban {len(COLUMNAS)}: {COLUMNAS}.')
    df = pd.DataFrame([f[:len(COLUMNAS)] for f in filas], columns=COLUMNAS)
    for col in COLUMNAS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['date'] = pd.to_datetime(df.open_time, unit='ms', utc=True).dt.strftime('%Y-%m-%d')
    return df


def audit(df):
    """Describe los problemas de la serie sin modificarla."""
    invalidos = []
    for col in PRECIOS + ['volume']:
        malos = ~np.isfinite(df[col]) | (df[col] <= 0 if col in PRECIOS else df[col] < 0)
        invalidos += [{'columna': col, 'fecha': d, 'valor': v} for d, v in zip(df.date[malos], df[col][malos])]
    orden = df.drop_duplicates('open_time', keep='last').sort_values('open_time')
    saltos = orden.open_time.diff()
    huecos = [{'desde': a, 'hasta': b, 'dias': float(s) / DIA_MS}
              for a, b, s in zip(orden.date.shift(), orden.date, saltos) if pd.notna(s) and s >= 2 * DIA_MS]
    incoherentes = df[(df.high < df[['open', 'close']].max(axis=1)) | (df.low > df[['open', 'close']].min(axis=1))]
    return {
        'filas': len(df),
        'primera': orden.date.iloc[0],
        'ultima': orden.date.iloc[-1],
        'duplicadas': int(df.open_time.duplicated().sum()),
        'desordenadas': int((df.open_time.diff() < 0).sum()),
        'invalidos': invalidos,
        'huecos': huecos,
        'desalineadas': int((orden.open_time % DIA_MS != 0).sum()),
        'incoherentes': incoherentes.date.tolist(),
    }


def clean(df):
    """Regla documentada: quitar duplicados y filas inválidas, y conservar el tramo continuo más largo.

    No se rellena ningún día. Las velas con otro corte diario quedan fuera porque no miden lo mismo.
    Devuelve el DataFrame y la bitácora de lo que cambió.
    """
    bitacora = []
    inicial = len(df)
    salida = df.sort_values('open_time', kind='stable').drop_duplicates('open_time', keep='last')
    if len(salida) != inicial:
        bitacora.append(f'{inicial - len(salida)} duplicada(s) eliminada(s), se conserva la última versión.')
    validas = np.isfinite(salida[PRECIOS]).all(axis=1) & (salida[PRECIOS] > 0).all(axis=1) & (salida.volume >= 0)
    if (~validas).any():
        bitacora.append(f'{int((~validas).sum())} fila(s) con precio no válido retirada(s): {salida.date[~validas].tolist()}.')
    salida = salida[validas]
    alineadas = salida.open_time % DIA_MS == 0
    if (~alineadas).any():
        bitacora.append(f'{int((~alineadas).sum())} vela(s) con otro corte diario retirada(s).')
    salida = salida[alineadas]
    if salida.empty:
        raise DataError('Después de limpiar no queda ninguna vela válida. Revisa la fuente de datos.')
    tramo = (salida.open_time.diff() != DIA_MS).cumsum()
    mejor = tramo.value_counts().idxmax()
    continuo = salida[tramo == mejor].reset_index(drop=True)
    if len(continuo) != len(salida):
        bitacora.append(f'{len(salida) - len(continuo)} vela(s) fuera del tramo continuo más largo '
                        f'({continuo.date.iloc[0]} a {continuo.date.iloc[-1]}).')
    return continuo, bitacora


def add_target(df):
    """y_t = P_{t+1} / P_t - 1. La última fila queda sin objetivo observable."""
    salida = df.copy()
    salida['y'] = salida.close.shift(-1) / salida.close - 1
    return salida


def dataset_hash(df):
    contenido = df[COLUMNAS].to_csv(index=False).encode()
    return hashlib.sha256(contenido).hexdigest()


def split_indices(n, tr=0.70, va=0.15):
    """Partición cronológica por orden de la fecha objetivo. No hay división aleatoria."""
    n_tr, n_va = int(n * tr), int(n * va)
    if n_tr < 1 or n_va < 1 or n - n_tr - n_va < 1:
        raise DataError(f'Ejemplos insuficientes para TR/VA/TE: {n}.')
    return {'TR': np.arange(0, n_tr), 'VA': np.arange(n_tr, n_tr + n_va), 'TE': np.arange(n_tr + n_va, n)}

In [ ]:
%%writefile bot/features.py
"""Variables del modelo. La misma definición se usa al entrenar y en el agente.

Cada indicador usa solo historia disponible hasta t. Los primeros valores quedan vacíos.
"""
from collections import namedtuple

import numpy as np
import pandas as pd

from bot.data import DataError

Ventanas = namedtuple('Ventanas', 'X y inicio emision objetivo')

# Qué mide cada indicador y su fórmula; el notebook y el panel muestran esta tabla.
INDICADORES = {
    'dist_sma20': 'Distancia al promedio de 20 cierres: cierre / SMA20 - 1',
    'dist_ema20': 'Distancia a la media exponencial de 20 cierres: cierre / EMA20 - 1',
    'vol20': 'Desviación muestral de 20 rendimientos diarios, sin anualizar',
    'rsi14': 'RSI de 14 con suavizado de Wilder, que arranca con la media de los primeros 14 cambios; 100 sin pérdidas, 50 sin movimiento',
    'macd_hist': 'MACD (EMA12 - EMA26) menos su señal EMA9, dividido entre el cierre',
    'bb_pos': 'Posición en bandas de Bollinger: (cierre - SMA20) / (2 desviaciones poblacionales de 20 cierres)',
    'atr14': 'ATR de 14 con suavizado de Wilder, dividido entre el cierre',
    'vol_rel': 'Logaritmo del volumen entre su promedio de 20 días',
}


def _wilder(serie, periodo):
    return serie.ewm(alpha=1 / periodo, adjust=False, min_periods=periodo).mean()


def _wilder_desde_media(serie, periodo):
    """Suavizado de Wilder del RSI: el primer valor es la media de los primeros `periodo` datos y después pesa 1/periodo."""
    x = serie.dropna()
    if len(x) < periodo:
        return pd.Series(np.nan, index=serie.index)
    semilla = pd.Series([x.iloc[:periodo].mean()], index=x.index[periodo - 1:periodo])
    return pd.concat([semilla, x.iloc[periodo:]]).ewm(alpha=1 / periodo, adjust=False).mean().reindex(serie.index)


def add_indicators(df):
    s = df.copy()
    cierre = s.close
    s['ret'] = cierre.pct_change(fill_method=None)
    s['sma20'] = cierre.rolling(20).mean()
    desv20 = cierre.rolling(20).std(ddof=0)  # las bandas de Bollinger usan la desviación poblacional de los 20 cierres
    s['dist_sma20'] = cierre / s.sma20 - 1
    s['dist_ema20'] = cierre / cierre.ewm(span=20, adjust=False, min_periods=20).mean() - 1
    s['vol20'] = s.ret.rolling(20).std(ddof=1)

    delta = cierre.diff()
    ganancia = _wilder_desde_media(delta.clip(lower=0), 14)
    perdida = _wilder_desde_media((-delta).clip(lower=0), 14)
    s['rsi14'] = (100 * ganancia / (ganancia + perdida)).mask((ganancia + perdida) == 0, 50.0)

    macd = cierre.ewm(span=12, adjust=False, min_periods=12).mean() - cierre.ewm(span=26, adjust=False, min_periods=26).mean()
    s['macd_hist'] = (macd - macd.ewm(span=9, adjust=False, min_periods=9).mean()) / cierre
    s['bb_pos'] = ((cierre - s.sma20) / (2 * desv20)).where(desv20 != 0, 0.0).where(s.sma20.notna())

    previo = cierre.shift()
    rango = pd.concat([s.high - s.low, (s.high - previo).abs(), (s.low - previo).abs()], axis=1).max(axis=1)
    s['atr14'] = _wilder(rango, 14) / cierre
    with np.errstate(divide='ignore', invalid='ignore'):
        s['vol_rel'] = np.log(s.volume / s.volume.rolling(20).mean())
    return s


def first_valid_dates(df):
    return {c: df.date[df[c].first_valid_index()] for c in INDICADORES if df[c].first_valid_index() is not None}


def _matriz(df, variables):
    faltantes = [v for v in variables if v not in df.columns]
    if faltantes:
        raise DataError(f'Variables inexistentes: {faltantes}. Disponibles: {list(df.columns)}.')
    return df[variables].to_numpy(dtype=float)


def make_windows(df, variables, lookback):
    """Ejemplos (n, pasos, variables). Cada uno termina en t y su objetivo es el rendimiento de t a t+1."""
    m = _matriz(df, variables)
    if 'y' not in df.columns:
        raise DataError('Falta la columna objetivo y: ejecuta add_target antes de construir ventanas.')
    y = df.y.to_numpy(dtype=float)
    X, Y, inicio, emision, objetivo = [], [], [], [], []
    for t in range(lookback - 1, len(df)):
        ventana = m[t - lookback + 1:t + 1]
        if np.isfinite(ventana).all() and np.isfinite(y[t]):
            X.append(ventana)
            Y.append(y[t])
            inicio.append(df.date.iloc[t - lookback + 1])
            emision.append(df.date.iloc[t])
            objetivo.append(df.date.iloc[t + 1])
    if not X:
        raise DataError(f'Historia insuficiente: {len(df)} filas no alcanzan para una ventana de {lookback} con {variables}.')
    return Ventanas(np.stack(X), np.array(Y), inicio, emision, objetivo)


def latest_window(df, variables, lookback):
    """Ventana de la última vela cerrada, para el pronóstico en vivo. Sin objetivo."""
    m = _matriz(df, variables)
    if len(m) < lookback:
        raise DataError(f'Historia insuficiente: hay {len(m)} velas y la ventana pide {lookback}.')
    x = m[-lookback:]
    if not np.isfinite(x).all():
        malas = [v for i, v in enumerate(variables) if not np.isfinite(x[:, i]).all()]
        raise DataError(f'Valores no finitos en {malas} dentro de la última ventana.')
    return x[None, ...], df.date.iloc[-1]


class Scaler:
    """Estandariza cada variable. Se ajusta solo con TR y después únicamente transforma."""

    def __init__(self, media, desv):
        self.media = np.asarray(media, dtype=float)
        self.desv = np.asarray(desv, dtype=float)

    @classmethod
    def fit(cls, X):
        if X.ndim != 3:
            raise DataError(f'Forma recibida {X.shape}; esperada (ejemplos, pasos, variables).')
        desv = X.std(axis=(0, 1))
        return cls(X.mean(axis=(0, 1)), np.where(desv > 0, desv, 1.0))

    def transform(self, X):
        if X.ndim != 3 or X.shape[2] != len(self.media):
            raise DataError(f'Forma recibida {X.shape}; esperada (ejemplos, pasos, {len(self.media)}).')
        return (X - self.media) / self.desv

    def to_dict(self):
        return {'media': self.media.tolist(), 'desv': self.desv.tolist()}

    @classmethod
    def from_dict(cls, d):
        return cls(d['media'], d['desv'])

In [ ]:
%%writefile bot/metricas_riesgo.py
"""Rendimiento y riesgo de una serie de patrimonio: volatilidad anual, Sharpe, Sortino, VaR y CVaR históricos.

Los rendimientos son simples y en decimales. Se anualiza con 365 periodos porque MEXC cotiza todos los días.
Sharpe y Sortino usan una tasa libre de riesgo y una meta anuales efectivas (0 por defecto), convertidas a su
equivalente diario. VaR y CVaR son históricos, a un periodo, y se reportan como pérdida positiva.
"""
import numpy as np

PERIODOS = 365


def var_cvar(rendimientos, c=0.95):
    """VaR: cuantil c de las pérdidas por inversa de la acumulada. CVaR: promedio de la fracción 1 - c de pérdidas
    más severas, con peso fraccionario en la última para conservar la masa exacta de la cola."""
    perdidas = np.sort(-np.asarray(rendimientos, dtype=float))[::-1]
    masa = (1 - c) * len(perdidas)
    pesos = np.clip(masa - np.arange(len(perdidas)), 0, 1)
    return float(np.quantile(perdidas, c, method='inverted_cdf')), float(perdidas @ pesos / masa)


def metricas_riesgo(patrimonio, tasa_anual=0.0, meta_anual=0.0, c=0.95, periodos=PERIODOS):
    """Con menos de dos rendimientos no hay dispersión que medir. Con desviación cero, Sharpe o Sortino quedan en None."""
    serie = np.asarray(patrimonio, dtype=float)
    r = serie[1:] / serie[:-1] - 1
    vacio = dict(observaciones=len(r), volatilidad_anual=None, sharpe=None, sortino=None, var=None, cvar=None)
    if len(r) < 2 or not np.isfinite(r).all():
        return vacio
    rf = (1 + tasa_anual) ** (1 / periodos) - 1
    meta = (1 + meta_anual) ** (1 / periodos) - 1
    exceso = r - rf
    desv = float(exceso.std(ddof=1))
    baja = float(np.sqrt(np.mean(np.minimum(r - meta, 0) ** 2)))
    var, cvar = var_cvar(r, c)
    raiz = np.sqrt(periodos)
    return dict(observaciones=len(r), volatilidad_anual=desv * raiz, sharpe=float(exceso.mean() / desv * raiz) if desv > 0 else None,
                sortino=float((r.mean() - meta) / baja * raiz) if baja > 0 else None, var=var, cvar=cvar)

In [ ]:
%%writefile bot/model.py
"""Inferencia del modelo aprobado con NumPy. El agente no necesita Keras, GPU ni servicios externos."""
import json
import math
import zipfile
from pathlib import Path

import numpy as np

from bot.data import DataError
from bot.features import Scaler, latest_window

TIPOS = ('cero', 'lineal', 'mlp', 'lstm')
FILTROS = ('ninguno', 'sma20')  # los mismos que evalúa research/lab.py
ACTIVACIONES = {'linear': lambda x: x, 'relu': lambda x: np.maximum(x, 0), 'tanh': np.tanh,
                'sigmoid': lambda x: 1 / (1 + np.exp(-x))}


def _sigmoide(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def _lstm(x, w, wr, b):
    """Celda LSTM con las mismas ecuaciones que Keras. Pesos en orden: entrada, olvido, celda, salida."""
    n, pasos, _ = x.shape
    unidades = wr.shape[0]
    h, c = np.zeros((n, unidades)), np.zeros((n, unidades))
    for t in range(pasos):
        z = x[:, t, :] @ w + h @ wr + b
        zi, zf, zc, zo = np.split(z, 4, axis=1)
        i, f, o = _sigmoide(zi), _sigmoide(zf), _sigmoide(zo)
        c = f * c + i * np.tanh(zc)
        h = o * np.tanh(c)
    return h


def adelante(miembro, pesos, indice, x):
    """Propagación hacia adelante de un miembro. x: (ejemplos, pasos, variables)."""
    salida = x
    for j, capa in enumerate(miembro):
        prefijo = f'm{indice}_c{j}_'
        if capa['tipo'] == 'lstm':
            salida = _lstm(salida, pesos[prefijo + 'w'], pesos[prefijo + 'wr'], pesos[prefijo + 'b'])
        else:
            if salida.ndim == 3:
                salida = salida.reshape(len(salida), -1)
            salida = ACTIVACIONES[capa['activacion']](salida @ pesos[prefijo + 'w'] + pesos[prefijo + 'b'])
    return salida.ravel()


def exportar_keras(red, indice):
    """Convierte una red Keras en lista de capas y pesos NumPy. Dropout no actúa al predecir."""
    miembro, pesos = [], {}
    for capa in red.layers:
        nombre = type(capa).__name__
        if nombre in ('Dropout', 'InputLayer'):
            continue
        j = len(miembro)
        prefijo = f'm{indice}_c{j}_'
        if nombre == 'LSTM':
            w, wr, b = capa.get_weights()
            pesos |= {prefijo + 'w': w, prefijo + 'wr': wr, prefijo + 'b': b}
            miembro.append({'tipo': 'lstm'})
        elif nombre == 'Dense':
            w, b = capa.get_weights()
            pesos |= {prefijo + 'w': w, prefijo + 'b': b}
            miembro.append({'tipo': 'dense', 'activacion': capa.get_config()['activation']})
        else:
            raise DataError(f'Capa {nombre} sin exportación a NumPy.')
    return miembro, pesos


def filtro_efectivo(filtro, minimo=''):
    """El filtro de la regla de VA; si no trae ninguno, el mínimo de config/riesgo.json ('' la deja sin filtro)."""
    return (minimo or filtro) if filtro in (None, 'ninguno') else filtro


class Modelo:
    def __init__(self, contrato, pesos):
        self.contrato, self.pesos = contrato, pesos
        self.version = contrato['version']
        self.tipo = contrato['tipo']
        self.lookback = contrato['lookback']
        self.variables = contrato['variables']
        self.umbral = contrato['umbral']
        self.filtro = contrato['filtro']
        self.escalador = Scaler.from_dict(contrato['escalador'])

    def predecir_x(self, x):
        """Pronóstico en la escala original del rendimiento."""
        if self.tipo == 'cero':
            return np.zeros(len(x))
        escalado = self.escalador.transform(x)
        salidas = [adelante(m, self.pesos, i, escalado) for i, m in enumerate(self.contrato['miembros'])]
        return self.contrato['y_media'] + self.contrato['y_desv'] * np.mean(salidas, axis=0)

    def pronosticar(self, tabla):
        if self.tipo == 'cero':
            return 0.0
        x, _ = latest_window(tabla, self.variables, self.lookback)
        return float(self.predecir_x(x)[0])

    def senal(self, pronostico, tabla, filtro_minimo=''):
        """Regla elegida en VA: comprar si el pronóstico supera el umbral y el filtro técnico lo confirma.
        Si la regla no trae filtro, se usa `filtro_minimo` (decisión de Alex, 2026-11-13: ningún activo compra sin filtro)."""
        if not np.isfinite(pronostico):
            raise DataError('Pronóstico no finito: no se puede convertir en señal de trading.')
        ultima = tabla.iloc[-1]
        filtro = filtro_efectivo(self.filtro, filtro_minimo)
        cumple_umbral = bool(pronostico > self.umbral)
        cumple_filtro = True
        if filtro == 'sma20':
            cumple_filtro = bool(ultima.close > ultima.sma20)
        return cumple_umbral and cumple_filtro, dict(umbral=self.umbral, filtro=filtro, cumple_umbral=cumple_umbral,
                                                     cumple_filtro=cumple_filtro)


class ModeloInvalido:
    """Lo que recibe el agente cuyo modelo no pasó la revisión al cargar: queda en pausa y nunca pronostica."""

    def __init__(self, error):
        self.error = error


def _finito(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x)


def _revisar(contrato, pesos, version):
    """Integridad antes de operar: versión, contrato, escalador y pesos coherentes y finitos. Devuelve el problema o None."""
    if not isinstance(contrato.get('version'), str) or (version is not None and contrato['version'] != version):
        return f'la versión del contrato ({contrato.get("version")}) no coincide con la indicada ({version})'
    variables, lookback = contrato.get('variables'), contrato.get('lookback')
    if not (isinstance(variables, list) and variables and all(isinstance(v, str) for v in variables)):
        return 'la lista de variables está vacía o no es válida'
    if not (isinstance(lookback, int) and not isinstance(lookback, bool) and lookback >= 1):
        return f'la ventana (lookback) no es un entero positivo: {lookback}'
    if contrato.get('filtro') not in FILTROS:
        return f'filtro desconocido: {contrato.get("filtro")}'
    for clave in ('umbral', 'y_media', 'y_desv'):
        if not _finito(contrato.get(clave)):
            return f'{clave} no es un número finito: {contrato.get(clave)}'
    if contrato['y_desv'] <= 0:
        return 'y_desv debe ser positivo'
    escalador = contrato.get('escalador') if isinstance(contrato.get('escalador'), dict) else {}
    try:
        media, desv = np.asarray(escalador.get('media'), dtype=float), np.asarray(escalador.get('desv'), dtype=float)
    except (TypeError, ValueError):
        return 'el escalador no es numérico'
    if media.shape != (len(variables),) or desv.shape != media.shape:
        return f'el escalador no trae un valor por variable ({len(variables)})'
    if not (np.isfinite(media).all() and np.isfinite(desv).all() and (desv > 0).all()):
        return 'el escalador tiene valores no finitos o desviaciones no positivas'
    for k, w in pesos.items():
        if not (np.issubdtype(w.dtype, np.number) and np.isfinite(w).all()):
            return f'el peso {k} tiene valores no finitos o no numéricos'
    if contrato['tipo'] == 'cero':
        return None
    if not contrato.get('miembros'):
        return 'el contrato no declara miembros'
    try:  # prueba de humo: toda la red debe dar un número finito con una ventana neutra
        salida = Modelo(contrato, pesos).predecir_x(np.zeros((1, lookback, len(variables))))
    except (KeyError, ValueError, IndexError, TypeError) as e:
        return f'los pesos no encajan con el contrato ({type(e).__name__}: {e})'
    if salida.shape != (1,) or not np.isfinite(salida).all():
        return 'la red no produce un pronóstico finito'
    return None


def cargar(carpeta, version=None):
    """Carga y revisa el modelo. Con `version` (el nombre de ACTIVO.txt o CARTERA.txt), el contrato debe declarar esa misma versión."""
    carpeta = Path(carpeta)
    ruta = carpeta / 'contrato.json'
    if not ruta.exists():
        raise DataError(f'No existe {ruta}: copia ahí el modelo aprobado o genera uno con research/entrenar_cpu.py.')
    try:
        contrato = json.loads(ruta.read_text(encoding='utf-8'))
    except (OSError, ValueError) as e:
        raise DataError(f'Modelo {carpeta.name} inválido: contrato.json no se puede leer ({type(e).__name__}).') from e
    if not isinstance(contrato, dict):
        raise DataError(f'Modelo {carpeta.name} inválido: contrato.json no es un objeto.')
    if contrato.get('tipo') not in TIPOS:
        raise DataError(f'Tipo de modelo no soportado: {contrato.get("tipo")}. Permitidos: {TIPOS}.')
    pesos = {}
    if contrato['tipo'] != 'cero':
        npz = carpeta / 'modelo.npz'
        if not npz.exists():
            raise DataError(f'Falta {npz}: el contrato pide pesos y no están en la carpeta.')
        try:
            with np.load(npz) as d:
                pesos = {k: d[k] for k in d.files}
        except (OSError, ValueError, EOFError, zipfile.BadZipFile) as e:
            raise DataError(f'Modelo {carpeta.name} inválido: modelo.npz no se puede leer ({type(e).__name__}).') from e
    problema = _revisar(contrato, pesos, version)
    if problema:
        raise DataError(f'Modelo {carpeta.name} inválido: {problema}. No opera hasta reemplazarlo y reiniciar el bot.')
    return Modelo(contrato, pesos)

In [ ]:
%%writefile research/lab.py
"""Laboratorio: preparación, modelos, evaluación y simulación.

El mismo código corre en el notebook de Colab con T4 y en la laptop con CPU. TE solo se abre con abrir_te.
"""
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from bot import data, features, metricas_riesgo, model

UMBRALES = (0.0005, 0.001, 0.002, 0.004, 0.008)  # definidos antes de mirar VA; positivos y distintos del costo
FILTROS = ('ninguno', 'sma20')
CONJUNTOS_VARIABLES = {
    'rezagos': ['ret'],
    'rezagos_indicadores': ['ret', 'dist_sma20', 'vol20'],
}
CONFIGS_MLP = ({'hu': 16, 'lr': 1e-3}, {'hu': 32, 'lr': 1e-3}, {'hu': 32, 'lr': 3e-4})
CONFIGS_LSTM = ({'hu': 8, 'lr': 1e-3}, {'hu': 16, 'lr': 1e-3}, {'hu': 16, 'lr': 3e-4})
SEMILLAS = (11, 22, 33)


# Métricas
def _plano(a, b):
    a, b = np.asarray(a, dtype=float).reshape(-1), np.asarray(b, dtype=float).reshape(-1)
    if len(a) != len(b):
        raise ValueError(f'y y p deben tener la misma longitud: {len(a)} y {len(b)}.')
    if not (np.isfinite(a).all() and np.isfinite(b).all()):
        raise ValueError('y y p deben contener valores finitos.')
    return a, b


def metricas(y, p):
    """Escala original. error = estimado - real. En precisión direccional se excluyen los pares con un cero."""
    y, p = _plano(y, p)
    error = p - y
    validos = (y != 0) & (p != 0)
    largos, cortos = int((p > 0).sum()), int((p < 0).sum())
    denominador = float(((y - y.mean()) ** 2).sum())
    constante = np.std(p) == 0 or np.std(y) == 0
    return dict(MAE=float(np.abs(error).mean()), RMSE=float(np.sqrt((error ** 2).mean())),
                direccional=float((np.sign(y[validos]) == np.sign(p[validos])).mean()) if validos.any() else None,
                R2=1 - float((error ** 2).sum()) / denominador if denominador else None,
                correlacion=None if constante else float(np.corrcoef(y, p)[0, 1]),
                sesgo_modelo=largos / cortos if cortos else None, n=len(y))


def casi_constante(p):
    return bool(np.std(np.asarray(p, dtype=float)) < 1e-5)


def max_drawdown(patrimonio):
    serie = np.asarray(patrimonio, dtype=float)
    picos = np.maximum.accumulate(serie)
    return float(((picos - serie) / picos).max())


# Simulación de la estrategia
def simular(velas, senales, comision, capital=100_000.0):
    """La señal del cierre de t se ejecuta en la apertura de t+1. Solo posición comprada o efectivo."""
    senales = np.asarray(senales, dtype=bool)
    if len(senales) != len(velas):
        raise ValueError(f'{len(senales)} señales para {len(velas)} velas.')
    apertura, cierre = velas.open.to_numpy(dtype=float), velas.close.to_numpy(dtype=float)
    efectivo, cantidad, costos, entrada = float(capital), 0.0, 0.0, 0.0
    resultados, filas = [], []
    for t in range(len(velas)):
        comprar = t > 0 and senales[t - 1]
        ejecucion = None
        if comprar and cantidad == 0:
            nocional = efectivo / (1 + comision)  # incluye el costo al calcular cuánto se puede adquirir
            cantidad = nocional / apertura[t]
            costos += nocional * comision
            entrada, efectivo, ejecucion = efectivo, 0.0, apertura[t]
        elif not comprar and cantidad > 0:
            nocional = cantidad * apertura[t]
            costos += nocional * comision
            efectivo = nocional * (1 - comision)
            resultados.append(efectivo - entrada)
            cantidad, ejecucion = 0.0, apertura[t]
        filas.append(dict(fecha=velas.date.iloc[t], precio_ejecucion=ejecucion, senal=bool(senales[t]), posicion=cantidad,
                          efectivo=efectivo, costos=costos, patrimonio=efectivo + cantidad * cierre[t]))
    if cantidad > 0:  # liquidación final para comparar el cierre del periodo
        nocional = cantidad * cierre[-1]
        costos += nocional * comision
        efectivo = nocional * (1 - comision)
        resultados.append(efectivo - entrada)
        filas[-1] |= dict(precio_ejecucion=cierre[-1], posicion=0.0, efectivo=efectivo, costos=costos, patrimonio=efectivo)
    tabla = pd.DataFrame(filas)
    final = float(tabla.patrimonio.iloc[-1])
    return dict(patrimonio_final=final, rendimiento=final / capital - 1, drawdown=max_drawdown(tabla.patrimonio),
                operaciones=len(resultados), win_rate=float(np.mean([r > 0 for r in resultados])) if resultados else None,
                costos=costos, **metricas_riesgo.metricas_riesgo(np.r_[capital, tabla.patrimonio.to_numpy()]), tabla=tabla)


def comprar_y_mantener(velas, comision, capital=100_000.0):
    """Referencia pasiva desde la misma primera fecha de ejecución y con costos equivalentes."""
    return simular(velas, np.ones(len(velas), dtype=bool), comision, capital)


def elegir_regla(resultados):
    """Mayor rendimiento neto en VA; en empate, menor drawdown."""
    return max(resultados, key=lambda r: (round(r['rendimiento'], 10), -r['drawdown']))


# Partición y preparación
def verificar_particion(fechas_objetivo):
    if not max(fechas_objetivo['TR']) < min(fechas_objetivo['VA']):
        raise ValueError('El último objetivo de TR no es anterior al primero de VA: invalida el experimento.')
    if not max(fechas_objetivo['VA']) < min(fechas_objetivo['TE']):
        raise ValueError('El último objetivo de VA no es anterior al primero de TE: invalida el experimento.')


def folds_expansivos(n_tr, k=3):
    """Ventana expansiva dentro de TR: cada entrenamiento termina antes del bloque que evalúa."""
    bloque = n_tr // (k + 2)
    inicios = [n_tr - (k - i) * bloque for i in range(k)]
    return [(np.arange(0, a), np.arange(a, a + bloque)) for a in inicios]


def tabla_comun(tabla, variables=('ret', 'dist_sma20', 'vol20')):
    """Recorta el inicio para que todas las versiones se evalúen exactamente en las mismas fechas."""
    validas = tabla[list(variables)].notna().all(axis=1)
    return tabla[validas.idxmax():].reset_index(drop=True)


def preparar(tabla, variables, lookback=20):
    """Ventanas, partición por fecha objetivo y escalado ajustado solo con TR."""
    v = features.make_windows(tabla, variables, lookback)
    p = data.split_indices(len(v.y))
    fechas = {k: [v.objetivo[i] for i in idx] for k, idx in p.items()}
    verificar_particion(fechas)
    escalador = features.Scaler.fit(v.X[p['TR']])
    y_media, y_desv = float(v.y[p['TR']].mean()), float(v.y[p['TR']].std())
    salida = dict(variables=variables, lookback=lookback, escalador=escalador, y_media=y_media, y_desv=y_desv, indices=p, ventanas=v)
    for k, idx in p.items():
        salida[f'X_{k}'] = escalador.transform(v.X[idx])
        salida[f'Xcrudo_{k}'] = v.X[idx]
        salida[f'y_{k}'] = v.y[idx]
        salida[f'emision_{k}'] = [v.emision[i] for i in idx]
        salida[f'objetivo_{k}'] = fechas[k]
    return salida


def tabla_particion(d):
    return pd.DataFrame([dict(conjunto=k, ejemplos=len(d[f'y_{k}']), primera_emision=d[f'emision_{k}'][0], ultima_emision=d[f'emision_{k}'][-1],
                              primer_objetivo=d[f'objetivo_{k}'][0], ultimo_objetivo=d[f'objetivo_{k}'][-1]) for k in ('TR', 'VA', 'TE')])


def fila_de_control(d, conjunto='VA', i=0):
    """Una fila completa para explicar por qué el modelo no conoce la respuesta al predecir."""
    indice = d['indices'][conjunto][i]
    v = d['ventanas']
    return dict(entrada_desde=v.inicio[indice], entrada_hasta=v.emision[indice], emision=v.emision[indice], objetivo=v.objetivo[indice],
                rendimiento_objetivo=float(v.y[indice]))


# Modelos clásicos
def ajustar_lineal(X, y, alpha=0.0):
    """Mínimos cuadrados con intercepto. alpha > 0 es Ridge (L2); el intercepto no se penaliza."""
    media_x, media_y = X.mean(axis=0), y.mean()
    Xc = X - media_x
    w = np.linalg.solve(Xc.T @ Xc + (alpha + 1e-9) * np.eye(X.shape[1]), Xc.T @ (y - media_y))
    return w, float(media_y - media_x @ w)


def plano(X):
    return X.reshape(len(X), -1)


def prediccion_lineal(d, conjunto, w, b):
    return plano(d[f'X_{conjunto}']) @ w + b


def ajustar_lineal_modelo(d, alpha=0.0):
    w, b = ajustar_lineal(plano(d['X_TR']), d['y_TR'], alpha)
    return dict(tipo='lineal', w=w, b=b, alpha=alpha)


# Redes (Keras). Se importa aquí para que el agente no dependa de esta librería.
def crear_mlp(entradas, hu, lr, hl=2, dropout=0.0, l2=0.0):
    import keras
    from keras import layers, regularizers
    r = regularizers.L2(l2) if l2 else None
    red = keras.Sequential([layers.Input((entradas,))])
    for _ in range(hl):
        red.add(layers.Dense(hu, activation='relu', kernel_regularizer=r))
        if dropout:
            red.add(layers.Dropout(dropout))
    red.add(layers.Dense(1, activation='linear'))
    red.compile(loss='mse', optimizer=keras.optimizers.Adam(lr))
    return red


def crear_lstm(pasos, variables, hu, lr, dropout=0.0):
    import keras
    from keras import layers
    red = keras.Sequential([layers.Input((pasos, variables)), layers.LSTM(hu)])
    if dropout:
        red.add(layers.Dropout(dropout))
    red.add(layers.Dense(1, activation='linear'))
    red.compile(loss='mse', optimizer=keras.optimizers.Adam(lr))
    return red


def entrenar(red, Xtr, ytr, Xva, yva, semilla, epocas=200, lote=64, paciencia=15):
    """Parada temprana con VA restaurando los mejores pesos. Detiene el proceso si la pérdida no es finita."""
    import keras
    keras.utils.set_random_seed(semilla)
    inicio = time.perf_counter()
    h = red.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epocas, batch_size=lote, shuffle=True, verbose=0,
                callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=paciencia, restore_best_weights=True),
                           keras.callbacks.TerminateOnNaN()])
    curvas = {k: [float(x) for x in v] for k, v in h.history.items()}
    if not np.isfinite(curvas['loss']).all():
        raise ValueError('Pérdida no finita durante el entrenamiento: revisa escalas, datos e hiperparámetros.')
    return dict(red=red, curvas=curvas, segundos=time.perf_counter() - inicio, mejor_epoca=int(np.argmin(curvas['val_loss'])) + 1,
                parametros=int(red.count_params()))


def construir_red(tipo, d, cfg):
    k = d['X_TR'].shape[2]
    if tipo == 'mlp':
        return crear_mlp(d['X_TR'].shape[1] * k, cfg['hu'], cfg['lr'], dropout=cfg.get('dropout', 0.0), l2=cfg.get('l2', 0.0), hl=cfg.get('hl', 2))
    return crear_lstm(d['X_TR'].shape[1], k, cfg['hu'], cfg['lr'], dropout=cfg.get('dropout', 0.0))


def entradas_red(tipo, X):
    return plano(X) if tipo == 'mlp' else X


def entrenar_red(tipo, d, cfg, semilla, **kw):
    import keras
    keras.utils.set_random_seed(semilla)
    red = construir_red(tipo, d, cfg)
    ytr = (d['y_TR'] - d['y_media']) / d['y_desv']
    yva = (d['y_VA'] - d['y_media']) / d['y_desv']
    return entrenar(red, entradas_red(tipo, d['X_TR']), ytr, entradas_red(tipo, d['X_VA']), yva, semilla, **kw)


def predecir_red(tipo, r, d, conjunto):
    salida = r['red'].predict(entradas_red(tipo, d[f'X_{conjunto}']), verbose=0).ravel()
    return d['y_media'] + d['y_desv'] * salida


# Experimentos
def registrar(experimentos, d, capitulo, variables, modelo, cfg, semilla, tr, va, segundos=0.0, dispositivo='cpu'):
    experimentos.append(dict(capitulo=capitulo, variables=variables, modelo=modelo, configuracion=json.dumps(cfg, ensure_ascii=False), semilla=semilla,
                             dispositivo=dispositivo, segundos=round(segundos, 2), **{f'{k}_TR': v for k, v in metricas(d['y_TR'], tr).items()},
                             **{f'{k}_VA': v for k, v in metricas(d['y_VA'], va).items()}))


def probar_referencias(d, variables, experimentos, dispositivo='cpu'):
    """Referencias: pronóstico cero, regresión lineal y Ridge con los mismos rezagos."""
    registrar(experimentos, d, 7, variables, 'cero', {}, None, np.zeros_like(d['y_TR']), np.zeros_like(d['y_VA']), dispositivo=dispositivo)
    candidatos = []
    for alpha in (0.0, 100.0, 1000.0):
        ajuste = ajustar_lineal_modelo(d, alpha)
        tr, va = prediccion_lineal(d, 'TR', ajuste['w'], ajuste['b']), prediccion_lineal(d, 'VA', ajuste['w'], ajuste['b'])
        registrar(experimentos, d, 7, variables, 'lineal' if not alpha else 'ridge', dict(alpha=alpha), None, tr, va, dispositivo=dispositivo)
        candidatos.append(dict(nombre=f'Lineal alpha={alpha:g}', tipo='lineal', ajuste=ajuste, pred_va=va, variables=variables))
    return candidatos


def probar_red(d, variables, tipo, configs, experimentos, curvas, dispositivo='cpu', semillas=SEMILLAS, **kw):
    """Redes: tres configuraciones elegidas por RMSE de VA; la mejor se repite con tres semillas."""
    mejor = None
    for cfg in configs:
        inicio = time.perf_counter()
        r = entrenar_red(tipo, d, cfg, semillas[0], **kw)
        tr, va = predecir_red(tipo, r, d, 'TR'), predecir_red(tipo, r, d, 'VA')
        registrar(experimentos, d, 8, variables, tipo, {**cfg, 'parametros': r['parametros'], 'mejor_epoca': r['mejor_epoca']}, semillas[0], tr, va, time.perf_counter() - inicio, dispositivo)
        rmse = metricas(d['y_VA'], va)['RMSE']
        if mejor is None or rmse < mejor[0]:
            mejor = (rmse, cfg)
    corridas = []
    for s in semillas:
        inicio = time.perf_counter()
        r = entrenar_red(tipo, d, mejor[1], s, **kw)
        tr, va = predecir_red(tipo, r, d, 'TR'), predecir_red(tipo, r, d, 'VA')
        registrar(experimentos, d, 8, variables, f'{tipo}_mejor', {**mejor[1], 'parametros': r['parametros'], 'mejor_epoca': r['mejor_epoca']}, s, tr, va, time.perf_counter() - inicio, dispositivo)
        corridas.append((r, va))
    rmses = [metricas(d['y_VA'], va)['RMSE'] for _, va in corridas]
    curvas[f'{variables}/{tipo}'] = corridas[0][0]['curvas']
    return dict(nombre=f'{tipo.upper()} ensamble de {len(semillas)} semillas, {mejor[1]["hu"]} neuronas', tipo=tipo, cfg=mejor[1], variables=variables, resultados=[r for r, _ in corridas],
                pred_va=np.mean([va for _, va in corridas], axis=0), rmse_media=float(np.mean(rmses)), rmse_desv=float(np.std(rmses)))


def validacion_expansiva(d, redes=(), **kw):
    """Validación temporal: tres ventanas dentro de TR. Las transformaciones se reajustan solo con cada entrenamiento."""
    v, indices, filas = d['ventanas'], d['indices']['TR'], []
    for n, (a, b) in enumerate(folds_expansivos(len(indices)), 1):
        X_ent, y_ent, X_blo, y_blo = v.X[indices[a]], v.y[indices[a]], v.X[indices[b]], v.y[indices[b]]
        escalador = features.Scaler.fit(X_ent)
        Xe, Xb = escalador.transform(X_ent), escalador.transform(X_blo)
        base = dict(ventana=n, entrenamiento_hasta=v.objetivo[indices[a][-1]], bloque_desde=v.objetivo[indices[b][0]], bloque_hasta=v.objetivo[indices[b][-1]],
                    ejemplos_entrenamiento=len(a), ejemplos_bloque=len(b))
        filas.append({**base, 'modelo': 'cero', **metricas(y_blo, np.zeros_like(y_blo))})
        w, c = ajustar_lineal(plano(Xe), y_ent)
        filas.append({**base, 'modelo': 'lineal', **metricas(y_blo, plano(Xb) @ w + c)})
        for tipo, cfg in redes:
            corte = int(len(Xe) * 0.85)  # la parada temprana usa el tramo final del propio entrenamiento
            y_media, y_desv = float(y_ent.mean()), float(y_ent.std())
            sub = dict(X_TR=Xe[:corte], y_TR=y_ent[:corte], X_VA=Xe[corte:], y_VA=y_ent[corte:], y_media=y_media, y_desv=y_desv)
            r = entrenar_red(tipo, sub, cfg, SEMILLAS[0], **kw)
            pred = y_media + y_desv * r['red'].predict(entradas_red(tipo, Xb), verbose=0).ravel()
            filas.append({**base, 'modelo': tipo, **metricas(y_blo, pred)})
    return filas


# Reglas de señal
def velas_de(tabla, fechas):
    indice = tabla.set_index('date')
    return indice.loc[fechas].reset_index()


def senales(pronostico, velas, umbral, filtro):
    pasa = np.asarray(pronostico) > umbral
    if filtro == 'sma20':
        pasa = pasa & (velas.close.to_numpy() > velas.sma20.to_numpy())
    return pasa


def probar_reglas(pronostico, velas, comision, capital=100_000.0):
    """Hasta cinco umbrales, con y sin filtro técnico. El filtro cambia operaciones, no las predicciones."""
    salida = []
    for filtro in FILTROS:
        for umbral in UMBRALES:
            r = simular(velas, senales(pronostico, velas, umbral, filtro), comision, capital)
            salida.append(dict(umbral=umbral, filtro=filtro, rendimiento=r['rendimiento'], drawdown=r['drawdown'], operaciones=r['operaciones'],
                               win_rate=r['win_rate'], costos=r['costos']))
    return salida


# Sesgo y varianza
def descomposicion(y, predicciones):
    """Sobre reentrenamientos: MSE = (sesgo² + ruido) + varianza. Con datos reales g(x) es desconocida,

    así que sesgo² y ruido solo se reportan juntos. La varianza sí se mide de verdad entre reentrenamientos.
    No confundir con MAE ni con la dispersión de una sola predicción.
    """
    y = np.asarray(y, dtype=float).reshape(-1)
    p = np.asarray(predicciones, dtype=float)
    if p.ndim != 2 or p.shape[1] != len(y):
        raise ValueError(f'Forma recibida {p.shape}; esperada (reentrenamientos, {len(y)}).')
    mse = float(((p - y) ** 2).mean())
    varianza = float(p.var(axis=0).mean())
    return dict(mse=mse, varianza=varianza, sesgo2_mas_ruido=mse - varianza, mse_promedio=float(((p.mean(axis=0) - y) ** 2).mean()), reentrenamientos=len(p))


def bootstrap_bloques(n, rng, largo=20):
    """Remuestreo por bloques para respetar la dependencia temporal al medir varianza."""
    inicios = rng.integers(0, n - largo + 1, size=n // largo + 1)
    return np.concatenate([np.arange(a, a + largo) for a in inicios])[:n]


def diagnostico_clasico(d, ajustar_predecir, reentrenos=12, semilla=7, fraccion=1.0):
    """Reentrena con remuestreos de TR y mide en VA. ajustar_predecir(Xtr, ytr, Xva) -> predicción."""
    rng = np.random.default_rng(semilla)
    Xtr, ytr, Xva = plano(d['X_TR']), d['y_TR'], plano(d['X_VA'])
    inicio = int(len(Xtr) * (1 - fraccion))  # "más datos": menos fracción usa solo el tramo reciente de TR
    Xtr, ytr = Xtr[inicio:], ytr[inicio:]
    pred = []
    for _ in range(reentrenos):
        idx = bootstrap_bloques(len(Xtr), rng)
        pred.append(ajustar_predecir(Xtr[idx], ytr[idx], Xva))
    return descomposicion(d['y_VA'], np.array(pred))


def tacticas_clasicas(d, reentrenos=12):
    """Tácticas de mitigación de la clase, medidas con estos datos. RF y boosting son diagnóstico, no se despliegan."""
    from sklearn.decomposition import PCA
    from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
    from sklearn.linear_model import Lasso
    from sklearn.pipeline import make_pipeline

    def lineal(alpha):
        def f(X, y, Xva):
            w, b = ajustar_lineal(X, y, alpha)
            return Xva @ w + b
        return f

    def sk(fabrica):
        def f(X, y, Xva):
            return fabrica().fit(X, y).predict(Xva)
        return f

    casos = [
        ('Base: regresión lineal', 'Referencia', lineal(0.0), {}),
        ('Agregar más datos (referencia con solo 50% de TR)', 'Agregar más datos', lineal(0.0), {'fraccion': 0.5}),
        ('Ridge L2 alpha=100', 'Regularización', lineal(100.0), {}),
        ('Ridge L2 alpha=1000', 'Regularización', lineal(1000.0), {}),
        ('Lasso L1', 'Regularización', sk(lambda: Lasso(alpha=1e-3, max_iter=5000)), {}),
        ('PCA 5 componentes + lineal', 'Selección de características (PCA)', sk(lambda: make_pipeline(PCA(5), Lasso(alpha=1e-9, max_iter=5000))), {}),
        ('Bagging: Random Forest', 'Bagging', sk(lambda: RandomForestRegressor(n_estimators=40, max_depth=4, n_jobs=-1, random_state=0)), {}),
        ('Boosting: gradiente', 'Boosting', sk(lambda: HistGradientBoostingRegressor(max_depth=2, max_iter=60, learning_rate=0.05, random_state=0)), {}),
    ]
    salida = []
    for nombre, estrategia, funcion, kw in casos:
        r = diagnostico_clasico(d, funcion, reentrenos=reentrenos, **kw)
        salida.append(dict(modelo=nombre, estrategia=estrategia, **r))
    return salida


def tacticas_redes(d, semillas=SEMILLAS, **kw):
    """Complejidad, dropout y L2 en redes: la varianza se mide entre semillas."""
    casos = [
        ('MLP pequeña', 'Referencia de red', 'mlp', dict(hu=8, lr=1e-3)),
        ('MLP grande (más capas y neuronas)', 'Aumentar complejidad', 'mlp', dict(hu=64, lr=1e-3, hl=3)),
        ('MLP grande + Dropout 0.3', 'Regularización', 'mlp', dict(hu=64, lr=1e-3, hl=3, dropout=0.3)),
        ('MLP grande + L2', 'Regularización', 'mlp', dict(hu=64, lr=1e-3, hl=3, l2=1e-3)),
        ('LSTM', 'Red recurrente', 'lstm', dict(hu=16, lr=1e-3)),
    ]
    salida = []
    for nombre, estrategia, tipo, cfg in casos:
        pred = [predecir_red(tipo, entrenar_red(tipo, d, cfg, s, **kw), d, 'VA') for s in semillas]
        salida.append(dict(modelo=nombre, estrategia=estrategia, **descomposicion(d['y_VA'], np.array(pred))))
    return salida


# Entorno del agente futuro (capítulo 10)
def transicion_ejemplo(efectivo, precio_ejecucion, cierre, comision):
    """Una transición calculada a mano: comprar con todo el efectivo y valorar al cierre. La recompensa es el cambio del patrimonio."""
    nocional = efectivo / (1 + comision)
    cantidad = nocional / precio_ejecucion
    costo = nocional * comision
    nuevo = cantidad * cierre
    return dict(cantidad=cantidad, costo=costo, patrimonio_nuevo=nuevo, recompensa=nuevo - efectivo)


def huella(configuracion):
    return hashlib.sha256(json.dumps(configuracion, sort_keys=True, default=str).encode()).hexdigest()


# Sello de apertura de TE
def abrir_te(ruta, huella):
    """TE se evalúa una sola vez con la configuración congelada. Cualquier cambio posterior queda declarado."""
    ruta = Path(ruta)
    sello = json.loads(ruta.read_text(encoding='utf-8')) if ruta.exists() else dict(huella=huella, aperturas=0, contaminado=False)
    sello['aperturas'] += 1
    if huella != sello['huella']:
        sello['contaminado'] = True
    ruta.write_text(json.dumps(sello, indent=2), encoding='utf-8')
    return sello


# Empaquetado del modelo aprobado
def contrato_lineal(d, ajuste):
    """Una regresión lineal se expresa como una capa densa sobre las variables ya escaladas y en y normalizada."""
    w = np.asarray(ajuste['w']).reshape(-1, 1) / d['y_desv']
    b = np.array([(ajuste['b'] - d['y_media']) / d['y_desv']])
    return [[{'tipo': 'dense', 'activacion': 'linear'}]], {'m0_c0_w': w, 'm0_c0_b': b}


def contrato_redes(resultados):
    """Un ensamble promedia varias semillas: bagging de redes para reducir la varianza."""
    miembros, pesos = [], {}
    for i, r in enumerate(resultados):
        m, p = model.exportar_keras(r['red'], i)
        miembros.append(m)
        pesos |= p
    return miembros, pesos


def guardar_modelo(carpeta, version, tipo, d, umbral, filtro, miembros, pesos, extra=None):
    carpeta = Path(carpeta)
    carpeta.mkdir(parents=True, exist_ok=True)
    contrato = dict(version=version, tipo=tipo, lookback=d['lookback'], variables=d['variables'], umbral=umbral, filtro=filtro,
                    escalador=d['escalador'].to_dict(), y_media=d['y_media'], y_desv=d['y_desv'], miembros=miembros, **(extra or {}))
    (carpeta / 'contrato.json').write_text(json.dumps(contrato, indent=2, ensure_ascii=False), encoding='utf-8')
    if pesos:
        np.savez(carpeta / 'modelo.npz', **pesos)
    return model.cargar(carpeta)

In [ ]:
%%writefile research/cartera.py
"""Cartera conjunta: datos, partición, regla elegida con VA y guardado. La usan research/entrenar_cartera.py (CPU) y el notebook de la cartera en T4.

Junta los ejemplos de todos los activos con las mismas fechas de corte. Las variables son relativas, así que son comparables entre activos.
Elige modelo y regla con VA simulando la cartera completa a cupos iguales, y reporta cómo le fue a cada activo. Nunca abre TE.
"""
import json
from pathlib import Path

import numpy as np
import pandas as pd

from bot import data, features
from research import lab

COSTO = 0.001
VARIABLES = lab.CONJUNTOS_VARIABLES['rezagos_indicadores']


def cargar_tablas(raiz):
    raiz = Path(raiz)
    universo = json.loads((raiz / 'config/universo.json').read_text(encoding='utf-8'))
    tablas = {}
    for s in universo['simbolos']:
        t = pd.read_csv(raiz / 'data/cartera' / f'{s}.csv')
        tablas[s] = lab.tabla_comun(data.add_target(features.add_indicators(t)))
    return tablas, universo


def preparar_conjunto(tablas, fin_tr=None, fin_va=None):
    """Sin cortes dados, toma los de BTC; con cortes dados (fechas objetivo), los respeta para comparar modelos en el mismo VA."""
    if fin_tr is None or fin_va is None:
        btc = lab.preparar(tablas['BTCUSDT'], VARIABLES)
        fin_tr, fin_va = btc['objetivo_TR'][-1], btc['objetivo_VA'][-1]
    partes = {k: dict(X=[], y=[], emision=[], objetivo=[], activo=[]) for k in ('TR', 'VA', 'TE')}
    for s, t in tablas.items():
        v = features.make_windows(t, VARIABLES, 20)
        for i, f in enumerate(v.objetivo):
            k = 'TR' if f <= fin_tr else 'VA' if f <= fin_va else 'TE'
            p = partes[k]
            p['X'].append(v.X[i]); p['y'].append(v.y[i]); p['emision'].append(v.emision[i]); p['objetivo'].append(f); p['activo'].append(s)
    tr = np.stack(partes['TR']['X'])
    escalador = features.Scaler.fit(tr)
    y_tr = np.array(partes['TR']['y'])
    d = dict(variables=VARIABLES, lookback=20, escalador=escalador, y_media=float(y_tr.mean()), y_desv=float(y_tr.std()), fin_tr=fin_tr, fin_va=fin_va)
    for k, p in partes.items():
        X = np.stack(p['X']) if p['X'] else np.zeros((0, 20, len(VARIABLES)))
        d[f'X_{k}'] = escalador.transform(X)
        d[f'Xcrudo_{k}'] = X
        d[f'y_{k}'] = np.array(p['y'])
        d[f'emision_{k}'] = p['emision']
        d[f'objetivo_{k}'] = p['objetivo']
        d[f'activo_{k}'] = np.array(p['activo'])
    return d


def simular_cartera(tablas, d, pronostico, umbral, filtro, capital=50_000.0):
    """Cupos iguales. Cada activo opera su regla; el patrimonio conjunto suma los cupos por fecha."""
    cupo = capital / len(tablas)
    series, por_activo = {}, []
    for s, t in tablas.items():
        mascara = d['activo_VA'] == s
        if not mascara.any():
            continue
        velas = lab.velas_de(t, [e for e, a in zip(d['emision_VA'], d['activo_VA']) if a == s])
        r = lab.simular(velas, lab.senales(pronostico[mascara], velas, umbral, filtro), COSTO, cupo)
        series[s] = r['tabla'].set_index('fecha').patrimonio
        m = lab.metricas(d['y_VA'][mascara], pronostico[mascara])
        pasivo = lab.comprar_y_mantener(velas, COSTO, cupo)
        por_activo.append(dict(simbolo=s, ejemplos=int(mascara.sum()), RMSE=m['RMSE'], RMSE_cero=lab.metricas(d['y_VA'][mascara], np.zeros(mascara.sum()))['RMSE'], direccional=m['direccional'],
                               rendimiento=r['rendimiento'], comprar_y_mantener=pasivo['rendimiento'], drawdown=r['drawdown'], operaciones=r['operaciones'], win_rate=r['win_rate'], costos=r['costos']))
    tabla = pd.DataFrame(series).sort_index().ffill().fillna(cupo)
    total = tabla.sum(axis=1) + (len(tablas) - tabla.shape[1]) * cupo
    return dict(rendimiento=float(total.iloc[-1] / capital - 1), drawdown=lab.max_drawdown(total.to_numpy()), operaciones=sum(a['operaciones'] for a in por_activo), costos=sum(a['costos'] for a in por_activo),
                ganadoras=np.mean([a['win_rate'] for a in por_activo if a['win_rate'] is not None]) if por_activo else None, por_activo=por_activo, serie=total)


def pasivo_cartera(tablas, d, capital=50_000.0):
    cupo = capital / len(tablas)
    series = {}
    for s, t in tablas.items():
        fechas = [e for e, a in zip(d['emision_VA'], d['activo_VA']) if a == s]
        if fechas:
            series[s] = lab.comprar_y_mantener(lab.velas_de(t, fechas), COSTO, cupo)['tabla'].set_index('fecha').patrimonio
    tabla = pd.DataFrame(series).sort_index().ffill().fillna(cupo)
    total = tabla.sum(axis=1) + (len(tablas) - tabla.shape[1]) * cupo
    return dict(rendimiento=float(total.iloc[-1] / capital - 1), drawdown=lab.max_drawdown(total.to_numpy()))


def seleccionar(tablas, d, candidatos):
    """Regla de cada candidato elegida con VA simulando la cartera completa; gana el mayor rendimiento y, en empate, la menor caída."""
    reglas = []
    for c in candidatos:
        c['casi_constante'] = lab.casi_constante(c['pred_va'])
        c['metricas_va'] = lab.metricas(d['y_VA'], c['pred_va'])
        filas = []
        for filtro in lab.FILTROS:
            for umbral in lab.UMBRALES:
                r = simular_cartera(tablas, d, c['pred_va'], umbral, filtro)
                filas.append(dict(umbral=umbral, filtro=filtro, rendimiento=r['rendimiento'], drawdown=r['drawdown'], operaciones=r['operaciones'], win_rate=r['ganadoras'], costos=r['costos']))
        c['regla'] = lab.elegir_regla(filas)
        reglas += [dict(variables='rezagos_indicadores', modelo=c['nombre'], **f) for f in filas]
        print(c['nombre'], 'RMSE', round(c['metricas_va']['RMSE'], 6), 'regla', c['regla'], flush=True)
    final = max(candidatos, key=lambda c: (round(c['regla']['rendimiento'], 10), -c['regla']['drawdown']))
    return final, reglas


def guardar(salida, version, tablas, d, final, reglas, experimentos, curvas, extra):
    """Contrato, pesos y tablas de VA en la carpeta del modelo. Comprueba que la exportación reproduce las predicciones."""
    salida = Path(salida)
    miembros, pesos = lab.contrato_lineal(d, final['ajuste']) if final['tipo'] == 'lineal' else lab.contrato_redes(final['resultados'])
    sim = simular_cartera(tablas, d, final['pred_va'], final['regla']['umbral'], final['regla']['filtro'])
    pasivo = pasivo_cartera(tablas, d)
    resumen = dict(periodo_va=[min(d['emision_VA']), max(d['emision_VA'])], modelo=final['nombre'], variables='rezagos_indicadores', regla=final['regla'], metricas_va=final['metricas_va'], comprar_y_mantener_va=pasivo,
                   rmse_cero_va=lab.metricas(d['y_VA'], np.zeros_like(d['y_VA']))['RMSE'], activos=len(tablas),
                   particion=[dict(conjunto=k, ejemplos=len(d[f'y_{k}']), primera_emision=min(d[f'emision_{k}']), ultima_emision=max(d[f'emision_{k}']), primer_objetivo=min(d[f'objetivo_{k}']), ultimo_objetivo=max(d[f'objetivo_{k}'])) for k in ('TR', 'VA', 'TE')])
    m = lab.guardar_modelo(salida, version, final['tipo'], d, final['regla']['umbral'], final['regla']['filtro'], miembros, pesos,
                           extra=dict(**extra, nombre=final['nombre'], conjunto_variables='rezagos_indicadores', seleccion=resumen, activos=list(tablas)))
    assert np.allclose(m.predecir_x(d['Xcrudo_VA']), final['pred_va'], atol=1e-4), 'La exportación no reproduce las predicciones.'
    pd.DataFrame(experimentos).to_csv(salida / 'experimentos.csv', index=False)
    pd.DataFrame(reglas).to_csv(salida / 'reglas_va.csv', index=False)
    pd.DataFrame(sim['por_activo']).to_csv(salida / 'por_activo_va.csv', index=False)
    (salida / 'curvas.json').write_text(json.dumps(curvas), encoding='utf-8')
    return sim, pasivo, resumen

In [ ]:
import numpy as np, pandas as pd
from urllib.parse import quote
from urllib.request import urlopen
from bot import data
from research import cartera, lab
keras.utils.set_random_seed(SEMILLA_GLOBAL)
np.random.seed(SEMILLA_GLOBAL)
COMMIT = '96f7faaf8692c979e3621983405da2d592a962f0'
FIN_TR, FIN_VA = '2024-08-02', '2025-09-17'  # mismos cortes que dev-cartera-2025-09-17-40: los dos modelos se comparan en el mismo VA y TE sigue cerrado
ENTRENAMIENTO = dict(epocas=2, lote=256, paciencia=1) if PRUEBA else dict(epocas=60, lote=256, paciencia=8)  # los mismos del entrenamiento en CPU

## 1. Datos de la cartera

In [ ]:
BASE = f'https://raw.githubusercontent.com/ianalexga15-hash/astrolabio/{COMMIT}/'
LOCAL = os.environ.get('NB_DATOS')  # solo al probar fuera de Colab: carpeta local del proyecto

def traer(ruta):
    destino = Path(ruta)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if LOCAL:
        shutil.copy(Path(LOCAL) / ruta, destino)
    else:
        destino.write_bytes(urlopen(BASE + quote(ruta), timeout=60).read())

traer('config/universo.json')
universo = json.loads(Path('config/universo.json').read_text(encoding='utf-8'))
if PRUEBA:
    universo['simbolos'] = universo['simbolos'][:4]
    Path('config/universo.json').write_text(json.dumps(universo), encoding='utf-8')
for s in universo['simbolos']:
    traer(f'data/cartera/{s}.csv')
huella_datos = lab.huella({s: data.dataset_hash(pd.read_csv(f'data/cartera/{s}.csv')) for s in universo['simbolos']})
tablas, _ = cartera.cargar_tablas('.')
d = cartera.preparar_conjunto(tablas, FIN_TR, FIN_VA)
assert max(d['objetivo_TR']) <= FIN_TR < min(d['objetivo_VA']) and max(d['objetivo_VA']) <= FIN_VA, 'La partición no respeta los cortes.'
display(pd.DataFrame([dict(conjunto=k, ejemplos=len(d[f'y_{k}']), primer_objetivo=min(d[f'objetivo_{k}']), ultimo_objetivo=max(d[f'objetivo_{k}'])) for k in ('TR', 'VA', 'TE')]))
print('Activos:', len(tablas), '| huella de los datos:', huella_datos[:16], '| TE no se evalúa en este notebook.')

## 2. Redes en la T4

Referencias (pronóstico cero, regresión lineal y Ridge), MLP y LSTM. Tres configuraciones por red elegidas por RMSE de VA; la mejor se repite con tres semillas y forma un ensamble. Parada temprana con VA.

In [ ]:
DISPOSITIVO = 'cpu (prueba)' if PRUEBA else evidencia['gpu']
experimentos, curvas = [], {}
inicio_redes = time.time()
candidatos = lab.probar_referencias(d, 'rezagos_indicadores', experimentos, dispositivo=DISPOSITIVO)
for tipo, configs in (('mlp', lab.CONFIGS_MLP), ('lstm', lab.CONFIGS_LSTM)):
    c = lab.probar_red(d, 'rezagos_indicadores', tipo, configs, experimentos, curvas, dispositivo=DISPOSITIVO, **ENTRENAMIENTO)
    print(c['nombre'], f"RMSE_VA media={c['rmse_media']:.6f}", f'{time.time() - inicio_redes:.0f} s en {DISPOSITIVO}', flush=True)
    candidatos.append(c)
if not PRUEBA:
    evidencia['memoria_gpu_pico'] = tf.config.experimental.get_memory_info('GPU:0')['peak']
display(pd.DataFrame(experimentos))

## 3. Regla elegida con VA, simulando la cartera completa

In [ ]:
final, reglas = cartera.seleccionar(tablas, d, candidatos)
print('Elegido:', final['nombre'], final['regla'])

## 4. Modelo, evidencia y descarga

In [ ]:
version = f"t4-cartera-{d['fin_va']}-{len(tablas)}" if not PRUEBA else 'prueba-cartera-sin-t4'
carpeta = SALIDA / version
motivo = 'Modelo conjunto entrenado en T4 con el protocolo de la cartera; TE sin abrir.' if not PRUEBA else 'Prueba del notebook sin T4.'
sim, pasivo, resumen = cartera.guardar(carpeta, version, tablas, d, final, reglas, experimentos, curvas,
                                       extra=dict(valido_para_entrega=not PRUEBA, dispositivo=DISPOSITIVO, motivo=motivo, huella_datos=huella_datos, commit_datos=COMMIT))
print('Modelo exportado y verificado: NumPy reproduce las predicciones de Keras en VA.')
print('Cartera en VA:', round(sim['rendimiento'], 4), 'caída', round(sim['drawdown'], 4), '| comprar y mantener:', round(pasivo['rendimiento'], 4), 'caída', round(pasivo['drawdown'], 4))
evidencia |= dict(fin=time.strftime('%Y-%m-%d %H:%M:%S'), segundos_totales=round(time.time() - inicio_notebook), dispositivo_entrenamiento=DISPOSITIVO)
(carpeta / 'evidencia_t4.json').write_text(json.dumps(evidencia, indent=2, ensure_ascii=False))
print(json.dumps(evidencia, indent=2, ensure_ascii=False))

In [ ]:
with zipfile.ZipFile('resultados_cartera_t4.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for ruta in SALIDA.rglob('*'):
        z.write(ruta, ruta.relative_to(SALIDA))
print('Listo:', Path('resultados_cartera_t4.zip').stat().st_size // 1024, 'KB. Carpeta del modelo:', version)
if not PRUEBA:
    from google.colab import files
    files.download('resultados_cartera_t4.zip')

## Cuándo reemplaza al modelo activo

Regla fijada antes de ver resultados: el modelo de T4 reemplaza al activo solo si, en el replay de la cartera sobre VA, sube el rendimiento sin que suba la caída máxima, o baja la caída sin que baje el rendimiento. Un resultado mixto lo deja como evidencia, sin activarlo. TE no se usa para decidir.